**Install Required Dependencies**

In [3]:
!pip install langchain-google-genai
!pip install beautifulsoup4
!pip install chromadb
!pip install selenium
!pip install webdriver_manager
!pip install requests
!pip install nest_asyncio
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.0/413.0 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.33
    Uninstalling langchain-core-0.3.33:
      Successfully uninstalled langchain-core-0.3.33
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.5
    Uninstalling langchain-text-splitters-0.3.5:
      Successfully uninstalled langchain-text-splitters-0.3.5
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.17
    Uninstalling langchain-0.3.17:
      Successfully uninstalled langchain-0.3.17


**Import Required Libraries**

In [4]:
import os
import requests
from bs4 import BeautifulSoup
from typing import List, Dict
import chromadb
from chromadb.utils import embedding_functions
import google.generativeai as genai
from langchain_google_genai import GoogleGenerativeAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import nest_asyncio
nest_asyncio.apply()

**Configure API Keys and Models**

In [5]:
GOOGLE_API_KEY = "Google_api_key" #replace with your gemini api key.
genai.configure(api_key=GOOGLE_API_KEY)

**Web Scraping Class**

In [6]:
class WebScraper:
    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

    def clean_text(self, text: str) -> str:
        """Clean and normalize scraped text."""
        import re
        # Remove extra whitespace and newlines
        text = ' '.join(text.split())
        # Remove special characters
        text = re.sub(r'[^\w\s.,!?-]', '', text)
        return text.strip()

    def scrape_website(self, url: str) -> List[Dict[str, str]]:
        """Scrape content from a website and return structured data."""
        try:
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # Remove unwanted elements
            for element in soup(['script', 'style', 'nav', 'footer']):
                element.decompose()

            # Extract main content
            content_blocks = []

            # Get headings and their associated content
            for heading in soup.find_all(['h1', 'h2', 'h3']):
                content = ""
                current = heading.find_next()

                while current and current.name not in ['h1', 'h2', 'h3']:
                    if current.name == 'p':
                        content += ' ' + current.get_text()
                    current = current.find_next()

                if content:
                    content_blocks.append({
                        'title': self.clean_text(heading.get_text()),
                        'content': self.clean_text(content),
                        'url': url
                    })

            # If no structured content found, extract all paragraphs
            if not content_blocks:
                all_text = ' '.join([p.get_text() for p in soup.find_all('p')])
                content_blocks.append({
                    'title': 'Main Content',
                    'content': self.clean_text(all_text),
                    'url': url
                })

            return content_blocks

        except Exception as e:
            print(f"Error scraping {url}: {str(e)}")
            return []

**Vector Database Manager**

In [7]:
class VectorDBManager:
    def __init__(self):
        # Initialize ChromaDB
        self.client = chromadb.Client()
        # Initialize text splitter
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
        )
        # Initialize embeddings
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2"
        )

    def create_collection(self, name: str):
        """Create a new ChromaDB collection."""
        return Chroma(
            collection_name=name,
            embedding_function=self.embeddings,
            persist_directory="./chroma_db"
        )

    def process_and_store(self, content_blocks: List[Dict[str, str]], collection_name: str):
        """Process content blocks and store in vector database."""
        vectorstore = self.create_collection(collection_name)

        for block in content_blocks:
            # Split content into chunks
            chunks = self.text_splitter.split_text(block['content'])

            # Create metadata for each chunk
            metadatas = [{
                'title': block['title'],
                'url': block['url'],
                'chunk_index': i
            } for i in range(len(chunks))]

            # Add documents to vector store
            vectorstore.add_texts(
                texts=chunks,
                metadatas=metadatas
            )

        return vectorstore

**LangChain Agent**

In [8]:
class WebQAAgent:
    def __init__(self, api_key: str):
        self.scraper = WebScraper()
        self.db_manager = VectorDBManager()
        self.llm = GoogleGenerativeAI(
            model="gemini-pro",
            google_api_key=api_key,
            temperature=0.3
        )

        self.qa_prompt = PromptTemplate(
            template="""Use the following pieces of context to answer the question at the end.
            If you don't know the answer, just say that you don't know, don't try to make up an answer.

            Context: {context}

            Question: {question}

            Please provide a detailed answer with relevant information from the context.
            If possible, cite specific parts of the context to support your answer.""",
            input_variables=["context", "question"]
        )

    def setup_website(self, url: str, collection_name: str):
        """Scrape website and set up vector database."""
        # Scrape content
        content_blocks = self.scraper.scrape_website(url)

        # Store in vector database
        vectorstore = self.db_manager.process_and_store(
            content_blocks,
            collection_name
        )

        # Create QA chain
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(
                search_kwargs={"k": 3}
            ),
            chain_type_kwargs={"prompt": self.qa_prompt}
        )

        return len(content_blocks)

    def answer_question(self, question: str) -> str:
        """Answer a question using the QA chain."""
        try:
            response = self.qa_chain.run(question)
            return response
        except Exception as e:
            return f"Error generating response: {str(e)}"


**Usage Example**

In [9]:
def main():
    # Initialize agent
    agent = WebQAAgent(GOOGLE_API_KEY)

    # Set up website
    url = "https://example.com"  # Replace with your target website
    num_blocks = agent.setup_website(url, "example_collection")
    print(f"Processed {num_blocks} content blocks from website")

    # Ask questions
    while True:
        question = input("Ask a question (or 'quit' to exit): ")
        if question.lower() == 'quit':
            break

        answer = agent.answer_question(question)
        print("\nAnswer:", answer)
        print("\n" + "="*50 + "\n")

if __name__ == "__main__":
    main()

<ipython-input-7-b6ef26497156>:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

<ipython-input-7-b6ef26497156>:19: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  return Chroma(


Processed 15 content blocks from website
Ask a question (or 'quit' to exit): What is LangChain


<ipython-input-8-49e1814c4fc1>:51: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = self.qa_chain.run(question)



Answer: LangChain is a framework for developing applications powered by language models. It provides a set of tools and components that make it easier to build complex AI applications, including features for document loading, text splitting, and integrating with various AI models and databases.

"LangChain is a framework for developing applications powered by language models. It provides a set of tools and components that make it easier to build complex AI applications, including features for document loading, text splitting, and integrating with various AI models and databases."


Ask a question (or 'quit' to exit): What is Ollama

Answer: Ollama is an open-source project that allows you to run large language models locally on your machine. It provides a simple interface for running and interacting with various AI models, making it easier to integrate advanced AI capabilities into your applications.

"Ollama is an open-source project that allows you to run large language models local